In [21]:
import numpy as np
import pandas as pd
import re
import string
import pickle

In [10]:
txt = 'great product i like it'

In [5]:
def remove_punctuation(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [6]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [7]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [8]:
def preprocess_text(text):
    data = pd.DataFrame( [text], columns=['tweet'])
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*', '', x, flags=re.MULTILINE) for x in x.split()))
    data["tweet"] = data ["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data["tweet"] = data["tweet"].apply(remove_punctuation)
    data["tweet"] = data ["tweet"].str.replace('\d+', '', regex=True)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(word) for word in x.split()))
    return data["tweet"].iloc[0]

<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Acer\AppData\Local\Temp\ipykernel_14016\1705870065.py:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  data["tweet"] = data ["tweet"].str.replace('\d+', '', regex=True)


In [11]:
preprocessed_txt = preprocess_text(txt)
print(preprocessed_txt)

great product like


In [15]:
vocab = pd.read_csv('../static/model/volcabulary.txt', header=None)
tokens = vocab[0].tolist()

In [17]:
def vectorizer(ds, volcabulary):
    vectorized_lst = []

    for sentence in ds:
        sentence_lst= np.zeros(len(volcabulary))
        
        for i in range(len(volcabulary)):
            if volcabulary[i] in sentence.split():
                sentence_lst[i] = 1

        vectorized_lst.append(sentence_lst)

    vectorized_lst_new=np.asarray(vectorized_lst, dtype=np.float32)
    return vectorized_lst_new

In [18]:
vectorized_txt = vectorizer([preprocessed_txt], tokens)

In [19]:
vectorized_txt

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 1145), dtype=float32)

In [24]:
with open('../static/model/sentiment_analysis_model.pkl', 'rb') as file:
    model = pickle.load(file)

In [25]:
model.predict(vectorized_txt)

array([1])

In [26]:
txt= 'this is a bad product'
preprocessed_txt = preprocess_text(txt)
vectorized_txt = vectorizer([preprocessed_txt], tokens)
model.predict(vectorized_txt)

array([1])

In [29]:
txt = 'good great awsome'
preprocessed_txt = preprocess_text(txt) 
vectorized_txt = vectorizer([preprocessed_txt], tokens)
model.predict(vectorized_txt)

array([0])

In [30]:
txt = 'awsom product..i love it'
preprocessed_txt = preprocess_text(txt)
vectorized_txt = vectorizer([preprocessed_txt], tokens)
model.predict(vectorized_txt)

array([0])

In [34]:
def get_prediction(vectorized_txt):
    prediction = model.predict(vectorized_txt)
    if prediction == 0:
        return 'positive'
    else:
        return 'negative'
    

In [35]:
txt = 'awesome product..i love it'
preprocessed_txt = preprocess_text(txt)
vectorized_txt = vectorizer([preprocessed_txt], tokens)
prediction = get_prediction(vectorized_txt)
prediction


'positive'